In [ ]:
!pip install datasets
!pip install torch
!pip install transformers

### Spacy dependencies
!pip install spacy
!python -m spacy download pt_core_news_sm

In [ ]:
import torch
from transformers.models.gemma3 import Gemma3ForCausalLM
import os
import re
import spacy
from datasets import Dataset, DatasetDict, load_dataset
import json
from tqdm import tqdm
from transformers import AutoTokenizer, Gemma3ForCausalLM, TrainingArguments, Trainer


GEMMA_PATH = "google/gemma-3-270m"
max_seq_length = 1024

os.environ["HF_TOKEN"] = "<HF_TOKEN>"

tokenizer = AutoTokenizer.from_pretrained(GEMMA_PATH, token=os.environ["HF_TOKEN"])

model = Gemma3ForCausalLM.from_pretrained(
    GEMMA_PATH,
    token=os.environ["HF_TOKEN"],
    dtype=torch.bfloat16,  
    device_map="auto",         
    attn_implementation="sdpa", 
    low_cpu_mem_usage=True       
)

special_tokens = [
    "<AGE>", "</AGE/>", "<PHONE>", "</PHONE/>", "<EMAIL>", "</EMAIL/>",
    "<DATE>", "</DATE/>", "<IDNUM>", "</IDNUM/>", "<MEDICAL_RECORD>", "</MEDICAL_RECORD/>",
    "<HEALTH_PLAN>", "</HEALTH_PLAN/>", "<STREET>", "</STREET/>", "<CITY>", "</CITY/>",
    "<ZIP>", "</ZIP/>", "<STATE>", "</STATE/>", "<COUNTRY>", "</COUNTRY/>",
    "<LOCATION_OTHER>", "</LOCATION_OTHER/>", "<ORGANIZATION>", "</ORGANIZATION/>",
    "<HOSPITAL>", "</HOSPITAL/>", "<PATIENT>", "</PATIENT/>", "<DOCTOR>", "</DOCTOR/>",
    "<USERNAME>", "</USERNAME/>", "<PROFESSION>", "</PROFESSION/>", "<OTHER>", "</OTHER/>"
]

tokenizer.add_tokens(special_tokens, special_tokens=True)
model.resize_token_embeddings(len(tokenizer))

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.generation_config.bos_token_id = tokenizer.bos_token_id

# Store the End-Of-Sequence token for use in prompt formatting
EOS_TOKEN = tokenizer.eos_token

In [ ]:
def fix_tags_with_replace(text):
    """
    Fixes the formatting of tags in the text using replace for each possible case.

    Args:
        text (str): The text generated by the model with malformed tags.

    Returns:
        str: Text with corrected tags.
    """
    # List of tags that need to be corrected
    tags = [
        "AGE", "PHONE", "FAX", "EMAIL", "URL", "IP_ADDRESS", "DATE", "IDNUM",
        "MEDICAL_RECORD", "DEVICE", "HEALTH_PLAN", "BIOID", "STREET", "CITY",
        "ZIP", "STATE", "COUNTRY", "LOCATION_OTHER", "ORGANIZATION", "HOSPITAL",
        "PATIENT", "DOCTOR", "USERNAME", "PROFESSION", "OTHER", "LOCATION"
    ]

    for tag in tags:
       # Fix spaces around the opening tag
        text = text.replace(f"< {tag} >", f"<{tag}>").replace(f"< {tag}>", f"<{tag}>").replace(f"<{tag} >", f"<{tag}>")
        # Fix spaces around the closing tag
        text = text.replace(f"</ {tag} >", f"</{tag}/>").replace(f"</ {tag}>", f"</{tag}/>").replace(f"</{tag} >", f"</{tag}/>")
        # Fix malformed closing tags with extra slashes
        text = text.replace(f"<{tag}/> ", f"</{tag}/>").replace(f"<{tag}/ >", f"</{tag}/>").replace(f"</{tag}/ >", f"</{tag}/>")
        # Remove spaces between tags and their inner content
        text = text.replace(f"<{tag}> ", f"<{tag}>").replace(f" </{tag}>", f"</{tag}/>").replace(f"</{tag}>", f"</{tag}/>")

    return text

def sliding_window(text, window_size=200, overlap=50):
    """
    Function that splits a text into sliding windows of size `window_size`
    with an overlap of `overlap`.

    Args:
        text (str): The text to be processed.
        window_size (int): The maximum number of tokens per window.
        overlap (int): The number of overlapping tokens between windows.

    Returns:
        List of str: List containing the text split into sliding windows.
    """
     # Load the SpaCy model for Portuguese (or another language, if needed)
    nlp = spacy.load('pt_core_news_sm')

    # Process the full text with SpaCy
    doc = nlp(text)

    # Extract the tokens
    tokens = [token.text for token in doc]

    # List to store the text windows
    windows = []

    # Sliding window implementation
    for i in range(0, len(tokens), window_size - overlap):
        # Capture a window of tokens
        window = tokens[i:i + window_size]
        windows.append(fix_tags_with_replace(" ".join(window)))

        # Stop if the text comes to a end
        if i + window_size >= len(tokens):
            break

    return windows

def preprocess_function(examples):
    prompts = [f"input_text: {re.sub(r'<[^>]+>', '', q)}\ntarget_text: {r} " + EOS_TOKEN  for q, r in zip(examples['text'], examples['text'])]
    tokenized_inputs = tokenizer(prompts, truncation=True, max_length=1024, padding="max_length")

    target = [f" {r} " + EOS_TOKEN  for r in examples['text']]
    tokenized_target = tokenizer(target, truncation=True, max_length=1024, padding="max_length")
    
    tokenized_inputs["labels"] = tokenized_target["input_ids"].copy()  # Important for causal LM
    return tokenized_inputs

preprocess_dataset = False

if preprocess_dataset:

    # Download dataset
    dataset_ = load_dataset("Venturus/AnonyMED-BR")

    train_chunks = [{'id': train_sample['id'], 'text': chunk, 'labels':'', 'synthetic':train_sample['synthetic']} for train_sample in tqdm(dataset_['train']) for chunk in sliding_window(train_sample["text"])]
    eval_chunks = [{'id': eval_sample['id'], 'text': chunk, 'labels':'', 'synthetic':eval_sample['synthetic']} for eval_sample in tqdm(dataset_['validation']) for chunk in sliding_window(eval_sample["text"])]

    with open('gemma_train.json', 'w', encoding='utf-8') as f: # Save intermediary step
        json.dump(train_chunks, f, ensure_ascii=False, indent=4)

    with open('gemma_eval.json', 'w', encoding='utf-8') as f: # Save intermediary step
        json.dump(eval_chunks, f, ensure_ascii=False, indent=4)

else:

    # Open and read the training set
    with open('gemma_train.json', 'r') as file:
        train_chunks = json.load(file)

    # Open and read the evaluation set
    with open('gemma_eval.json', 'r') as file:
        eval_chunks = json.load(file)

# Convert the data into Hugging Face Dataset objects
train_dataset_ = Dataset.from_list(train_chunks)
eval_dataset_ = Dataset.from_list(eval_chunks[0:500])

# Combine them into a DatasetDict
dataset = DatasetDict({
    "train": train_dataset_,
    "validation": eval_dataset_
})

train_dataset = dataset['train']
eval_dataset = dataset['validation']

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True)

In [ ]:
# # Training arguments
training_args = TrainingArguments(
    output_dir="Syn",
    dataloader_num_workers=0,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    save_strategy="epoch",
    logging_steps=200,
    report_to="none",
    warmup_ratio=0.03,
    fp16=False,
    push_to_hub=False,
    bf16=True,                         # Enable BFloat16
)

# Trainer setup and training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
)

trainer.train()


In [ ]:

## TEST
prompt_ = f"input_text: Subjetivo = = = = = = = = = = = = = = = = = = = = FISIOTERAPIA===================== 218 C - <DOCTOR>Rodrigo</DOCTOR/> <DOCTOR>Nascimento</DOCTOR/> Horário : 11:50 Neurológico : Paciente <PATIENT>Mariane</PATIENT/> <PATIENT>Silva</PATIENT/> , <AGE>60</AGE/> anos , <PROFESSION>fisioterapeuta</PROFESSION/> , lúcida , colaborativa , Glasgow 15 . Força muscular preservada . Hemodinâmico : Hemodinamicamente estável ( PAM 80 mmHg ) , FC de 88 bpm . Sem uso de aminas vasoativas . Respiratório : Em desmame ventilatório . MV+ bilateralmente , sem queixas respiratórias . Em CPAP com FiO2 de 28 % e pressão de 5 cmH2O. Anexos : Abdome normotenso , sem alterações na ausculta intestinal . Membros sem edema ou alteração de coloração . PERME : 5/5 MRC : NA Condutas : Continuar protocolo de desmame ventilatório . Realizar manobras de higiene brônquica e reavaliar em 2 horas . Elaborado e assinado por <DOCTOR>Antonio</DOCTOR/> <DOCTOR>Rodrigues</DOCTOR/> <DOCTOR>Alves</DOCTOR/> , Crefito <IDNUM>162859F</IDNUM/>.\ntarget_text:"

prompt =  re.sub(r'<[^>]+>', '', prompt_)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=1024)
answer = tokenizer.decode(outputs[0])

print(answer.split("target_text:")[1].split('<eos>')[0])

In [ ]:
# Save fine-tuned model
model.save_pretrained("<SAVE_PATH>")
tokenizer.save_pretrained("<SAVE_PATH>")